In [ ]:
import pandas as pd

from metrics.process_data import metrics_processor

In [ ]:
!python -m spacy download es_core_news_sm

In [ ]:
df = pd.read_excel("comparacion.xlsx", sheet_name="Matriz comparativa")

In [ ]:
df.iloc[28]

In [ ]:
df.iloc[28]["Original"]

In [ ]:
new_df = df.iloc[28].to_frame(name="prediction_text")
new_df.rename(columns={"28":"prediction_text"}, inplace=True)
new_df["original_text"] = df.iloc[28]["Original"]
new_df["simplified_text"] = df.iloc[28]["Lectura Fácil [Referencia]"]
new_df["document_id"] = new_df.index
new_df["original_sentence_id"] = new_df.index

In [ ]:
new_df.columns

In [ ]:
df_to_calculate_metric = new_df[4:]
df_to_calculate_metric = df_to_calculate_metric[
    ~df_to_calculate_metric['document_id'].str.contains('AdaptaTuTexto', na=False)
]

In [ ]:
# 2. Aplicar limpieza
for col in ['prediction_text', 'original_text', 'simplified_text']:
    if col in df_to_calculate_metric.columns:
        df_to_calculate_metric[col] = df_to_calculate_metric[col].apply(clean_text)

# 3. Filtro de longitud y exportación
df_to_calculate_metric = df_to_calculate_metric[df_to_calculate_metric['prediction_text'].str.len() > 5]

# Guardamos el archivo limpio para que MetricsProcessor lo use sin errores
df_to_calculate_metric.to_excel('clean.xlsx', index=False)

print(f"Limpieza finalizada. Filas listas para procesar: {len(df_to_calculate_metric)}")
df_to_calculate_metric.head()


In [ ]:
metric = metrics_processor(df_to_calculate_metric)
metric.save_consolidated_report("metrics.xlsx")